# 🧠 Derin Sinir Ağı Geliştirme — Çok-Hedefli Model (Multi-Output DNN)

Bu notebook, 4 kristal özelliğini **aynı anda** tahmin eden derin sinir ağının  
eğitim sürecini adım adım gösterir.

**Bu notebook ne işe yarar?**
- `train.py` dosyasının notebook versiyonudur — sonuçları görsel olarak takip edebilirsiniz
- K-Katlı Çapraz Doğrulama (K-Fold Cross Validation) eğitim döngüsü
- Her kat için kayıp (loss) eğrisi ve en iyi model seçimi
- Son modelin test seti üzerinde değerlendirilmesi

**Hedef değişkenler (aynı anda tahmin edilir):**
1. `formation_energy_per_atom` — Formasyon enerjisi (eV/atom)
2. `band_gap` — Bant aralığı (eV)
3. `cbm` — İletim bandı minimum enerjisi (eV)
4. `energy_above_hull` — Hull üstü enerji (eV/atom)

**Not:** Daha hızlı ve kapsamlı eğitim için `train.py` kullanmanız önerilir.  
Bu notebook, geliştirme sürecinin görsel belgelenmesi amacıyla hazırlanmıştır.

In [ ]:
# ============================================================
# KÜTÜPHANELER
# ============================================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader, Subset
from sklearn.model_selection import KFold
from sklearn.metrics import mean_absolute_error, r2_score

plt.rcParams['figure.dpi'] = 120

# GPU varsa kullan
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Eğitim cihazı: {device}')
if device.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')

In [ ]:
# ============================================================
# VERİ YÜKLEME — ÖN İŞLENMİŞ X ve y
# ============================================================
# preparation.py çalıştırıldıktan sonra oluşan dosyaları kullan.
# X: özellik matrisi (normaliz edilmemiş)
# y: 4 sütunlu hedef matrisi

X = pd.read_csv('data/X_preprocessed.csv')
y = pd.read_csv('data/y_preprocessed.csv')

target_names = y.columns.tolist()
n_targets    = len(target_names)
n_features   = X.shape[1]

print(f'Örnek sayısı : {X.shape[0]:,}')
print(f'Özellik sayısı: {n_features}')
print(f'Hedef sayısı  : {n_targets}  → {target_names}')

In [ ]:
# ============================================================
# TENSOR'LARA DÖNÜŞTÜRME VE NORMALİZASYON
# ============================================================
# Z-skoru normalizasyonu: (x - ortalama) / standart_sapma
# Bu, her özelliğin/hedefin aynı ölçeğe getirilmesini sağlar.
# Normalizasyon istatistikleri sonradan tahmin yaparken de gerekir — kaydet!

eps = 1e-8  # sıfıra bölmeyi önlemek için küçük epsilon

# X tensörü
X_tensor = torch.tensor(X.values, dtype=torch.float32).to(device)
X_mean = X_tensor.mean(dim=0)
X_std  = X_tensor.std(dim=0)
X_norm = (X_tensor - X_mean) / (X_std + eps)

# y tensörü — her hedef için ayrı ayrı normalize et
y_tensor = torch.tensor(y.values, dtype=torch.float32).to(device)
y_mean = y_tensor.mean(dim=0)  # shape [4]
y_std  = y_tensor.std(dim=0)   # shape [4]
y_norm = (y_tensor - y_mean) / (y_std + eps)

print('Normalizasyon tamamlandı.')
print(f'X aralığı sonrası: {X_norm.min().item():.2f} – {X_norm.max().item():.2f}')
print(f'y ortalamaları: {y_mean.cpu().numpy().round(3)}')
print(f'y std değerleri: {y_std.cpu().numpy().round(3)}')

In [ ]:
# ============================================================
# MODEL MİMARİSİ — ÇOK-HEDEFLİ DERİN SİNİR AĞI
# ============================================================
# Mimari: tam bağlantılı katmanlar (Fully Connected / Dense)
#
# Giriş boyutu : n_features (~346)
# Çıkış boyutu : n_targets (4)
#
# Katman yapısı:
#   Linear(n_features → 512) → ReLU → Dropout(0.1)
#   Linear(512 → 512)        → ReLU → Dropout(0.1)
#   Linear(512 → 256)        → ReLU
#   Linear(256 → 128)        → ReLU
#   Linear(128 → 64)         → ReLU
#   Linear(64 → 32)          → ReLU
#   Linear(32 → 4)           [çıkış — aktivasyon yok]
#
# Dropout: İlk iki ReLU'dan sonra %10 nöron geçici kapatma
#   → Aşırı öğrenmeyi (overfitting) azaltır
#
# Aktivasyon: ReLU (Rectified Linear Unit)
#   → Vanishing gradient sorununu azaltır, hesaplama açısından verimli

class NeuralNetwork(nn.Module):
    def __init__(self, input_size, output_size):
        super(NeuralNetwork, self).__init__()
        self.layers = nn.Sequential(
            nn.Linear(input_size, 512), nn.ReLU(), nn.Dropout(0.1),
            nn.Linear(512, 512),        nn.ReLU(), nn.Dropout(0.1),
            nn.Linear(512, 256),        nn.ReLU(),
            nn.Linear(256, 128),        nn.ReLU(),
            nn.Linear(128, 64),         nn.ReLU(),
            nn.Linear(64, 32),          nn.ReLU(),
            nn.Linear(32, output_size)
        )

    def forward(self, x):
        return self.layers(x)

# Model boyutunu test et
test_model = NeuralNetwork(n_features, n_targets)
toplam_param = sum(p.numel() for p in test_model.parameters())
print(f'Model başarıyla oluşturuldu.')
print(f'Toplam parametre sayısı: {toplam_param:,}')
print(test_model)

In [ ]:
# ============================================================
# K-KATLI ÇAPRAZ DOĞRULAMA EĞİTİMİ
# ============================================================
# K-Fold ile:
#   1. Veri k eşit parçaya bölünür
#   2. Her seferinde 1 parça test, geri kalanlar eğitim olarak kullanılır
#   3. k adet model eğitilir → genelleme performansı daha güvenilir ölçülür
#
# Erken durdurma (Early Stopping):
#   Test kaybı patience epoch boyunca iyileşmezse eğitim durdurulur.
#   Bu, gereksiz hesaplamayı önler ve aşırı öğrenmeyi azaltır.

# ─── Hiperparametreler ─────────────────────────────────────
K_FOLDS   = 5        # kaç kat
NUM_EPOCH = 100      # maksimum epoch sayısı (erken durdurma olabilir)
BATCH     = 256      # mini-batch boyutu
LR        = 1e-3     # başlangıç öğrenme hızı
PATIENCE  = 15       # erken durdurma sabrı
# ────────────────────────────────────────────────────────────

kfold = KFold(n_splits=K_FOLDS, shuffle=True, random_state=42)
dataset = TensorDataset(X_norm, y_norm)

kat_egitim_kayiplari = []
kat_test_kayiplari   = []
en_iyi_kayip         = float('inf')
en_iyi_agirliklar    = None

kayip_fn = nn.L1Loss()  # MAE (Ortalama Mutlak Hata) kaybı

for kat, (train_idx, test_idx) in enumerate(kfold.split(X_norm)):
    print(f'\n{"─"*50}')
    print(f'KAT {kat+1}/{K_FOLDS}  — Eğitim: {len(train_idx):,}  Test: {len(test_idx):,}')
    print(f'{"─"*50}')

    train_loader = DataLoader(
        Subset(dataset, train_idx), batch_size=BATCH, shuffle=True
    )
    test_loader = DataLoader(
        Subset(dataset, test_idx), batch_size=BATCH
    )

    # Her kat için yeni model — bağımsız başlat
    model = NeuralNetwork(n_features, n_targets).to(device)

    # Adam optimizeri: weight_decay ile L2 düzenlileştirme
    optimizer = optim.Adam(model.parameters(), lr=LR, weight_decay=1e-5)

    # Öğrenme hızı azaltma: test kaybı iyileşmezse LR'yi yarıya indir
    zamanlayici = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', factor=0.5, patience=5, verbose=True
    )

    en_iyi_kat_kayip = float('inf')
    sabir_sayaci = 0
    egitim_kayiplari, test_kayiplari = [], []

    for epoch in range(1, NUM_EPOCH + 1):
        # ── Eğitim adımı ──
        model.train()
        toplam_kayip = 0.0
        for X_batch, y_batch in train_loader:
            optimizer.zero_grad()
            tahmin = model(X_batch)
            kayip  = kayip_fn(tahmin, y_batch)
            kayip.backward()
            optimizer.step()
            toplam_kayip += kayip.item() * X_batch.size(0)
        egitim_kayip = toplam_kayip / len(train_loader.dataset)
        egitim_kayiplari.append(egitim_kayip)

        # ── Test adımı ──
        model.eval()
        toplam_test = 0.0
        with torch.no_grad():
            for X_batch, y_batch in test_loader:
                tahmin = model(X_batch)
                kayip  = kayip_fn(tahmin, y_batch)
                toplam_test += kayip.item() * X_batch.size(0)
        test_kayip = toplam_test / len(test_loader.dataset)
        test_kayiplari.append(test_kayip)

        zamanlayici.step(test_kayip)

        # En iyi modeli sakla (tüm katlar arasında)
        if test_kayip < en_iyi_kayip:
            en_iyi_kayip = test_kayip
            en_iyi_agirliklar = {k: v.clone() for k, v in model.state_dict().items()}

        # En iyi kat modeli sakla (erken durdurma için)
        if test_kayip < en_iyi_kat_kayip:
            en_iyi_kat_kayip = test_kayip
            sabir_sayaci = 0
        else:
            sabir_sayaci += 1
            if sabir_sayaci >= PATIENCE:
                print(f'  Erken durdurma: epoch {epoch} (sabır={PATIENCE})')
                break

        if epoch % 10 == 0:
            print(f'  Epoch {epoch:3d}/{NUM_EPOCH}  | Eğitim: {egitim_kayip:.4f}  | Test: {test_kayip:.4f}')

    kat_egitim_kayiplari.append(egitim_kayiplari[-1])
    kat_test_kayiplari.append(test_kayiplari[-1])

    # Her kat için kayıp eğrisi çiz
    plt.figure(figsize=(8, 4))
    plt.plot(egitim_kayiplari, label='Eğitim Kaybı')
    plt.plot(test_kayiplari, label='Test Kaybı', linestyle='--')
    plt.xlabel('Epoch'); plt.ylabel('L1 Kaybı (Normaliz.)')
    plt.title(f'Kat {kat+1} — Eğitim ve Test Kaybı', fontweight='bold')
    plt.legend(); plt.tight_layout()
    plt.savefig(f'figures/dnn_kat{kat+1}_egitim_kayip.png', dpi=120, bbox_inches='tight')
    plt.show()

print(f'\n{"═"*50}')
print('EĞİTİM TAMAMLANDI')
print(f'Ortalama Test Kaybı : {np.mean(kat_test_kayiplari):.4f} ± {np.std(kat_test_kayiplari):.4f}')
print(f'{"═"*50}')

In [ ]:
# ============================================================
# EN İYİ MODELİ KAYDET
# ============================================================
# Tüm katlar arasında en düşük test kaybını veren model ağırlıklarını kaydet.
# Normalizasyon istatistiklerini de kaydet — tahmin yaparken gerekecek.

import os
os.makedirs('models', exist_ok=True)

# En iyi model ağırlıkları
torch.save(en_iyi_agirliklar, 'models/best_model.pth')

# Normalizasyon istatistikleri — inference (tahmin) zamanında kullanılır
torch.save({
    'X_mean': X_mean.cpu(),
    'X_std':  X_std.cpu(),
    'y_mean': y_mean.cpu(),
    'y_std':  y_std.cpu(),
    'target_names': target_names,
    'n_targets': n_targets,
}, 'models/normalization_stats.pth')

print('En iyi model  → models/best_model.pth')
print('Normalizasyon → models/normalization_stats.pth')

In [ ]:
# ============================================================
# K-KAT BAZLI PERFORMANS ÖZETİ
# ============================================================

fig, ax = plt.subplots(figsize=(8, 5))
katlar = list(range(1, K_FOLDS + 1))
ax.plot(katlar, kat_egitim_kayiplari, 'o-', label='Eğitim Kaybı (son epoch)', color='steelblue')
ax.plot(katlar, kat_test_kayiplari,   's--', label='Test Kaybı (son epoch)', color='tomato')
ax.axhline(np.mean(kat_test_kayiplari), color='tomato', linestyle=':', lw=1.5,
           label=f'Ort. Test Kaybı: {np.mean(kat_test_kayiplari):.4f}')
ax.set_xlabel('Kat Numarası', fontsize=12)
ax.set_ylabel('L1 Kaybı (Normaliz. Uzay)', fontsize=12)
ax.set_title(f'{K_FOLDS}-Katlı Çapraz Doğrulama Sonuçları', fontsize=13, fontweight='bold')
ax.legend()
ax.set_xticks(katlar)
plt.tight_layout()
plt.savefig('figures/kfold_ozet.png', dpi=150, bbox_inches='tight')
plt.show()

print('K-Kat bazlı kayıplar:')
for k, (tr, te) in enumerate(zip(kat_egitim_kayiplari, kat_test_kayiplari), 1):
    print(f'  Kat {k}: Eğitim={tr:.4f}  Test={te:.4f}')

In [ ]:
# ============================================================
# HIZLI DEĞERLENDİRME — EN İYİ MODEL İLE TAHMİN
# ============================================================
# Kaydedilen en iyi modeli yükle ve tüm veri üzerinde tahmin yap.
# Her hedef için MAE ve R² göster.

# En iyi model ağırlıklarını yükle
en_iyi_model = NeuralNetwork(n_features, n_targets).to(device)
en_iyi_model.load_state_dict(en_iyi_agirliklar)
en_iyi_model.eval()

# Tüm örnekler üzerinde tahmin
with torch.no_grad():
    y_pred_norm = en_iyi_model(X_norm)
    y_pred_gercek = y_pred_norm * (y_std + eps) + y_mean  # denormalize

y_pred_np = y_pred_gercek.cpu().numpy()
y_true_np = y_tensor.cpu().numpy()

# Bant aralığı ve hull üstü enerji negatif olamaz
idx_bg   = target_names.index('band_gap')
idx_hull = target_names.index('energy_above_hull')
y_pred_np[:, idx_bg]   = np.clip(y_pred_np[:, idx_bg], 0, None)
y_pred_np[:, idx_hull] = np.clip(y_pred_np[:, idx_hull], 0, None)

print('═' * 60)
print(f'{'Hedef':<32} {'MAE':>9} {'R²':>8}')
print('═' * 60)
for i, ad in enumerate(target_names):
    mae = mean_absolute_error(y_true_np[:, i], y_pred_np[:, i])
    r2  = r2_score(y_true_np[:, i], y_pred_np[:, i])
    print(f'{ad:<32} {mae:>9.4f} {r2:>8.4f}')
print('═' * 60)
print('Daha ayrıntılı hata analizi için examine.ipynb defterine bakınız.')

## Model Mimarisi Özeti

| Katman | Giriş | Çıkış | Aktivasyon | Düzenlileştirme |
|--------|-------|-------|------------|-----------------|
| 1 | n_features | 512 | ReLU | Dropout(0.1) |
| 2 | 512 | 512 | ReLU | Dropout(0.1) |
| 3 | 512 | 256 | ReLU | — |
| 4 | 256 | 128 | ReLU | — |
| 5 | 128 | 64 | ReLU | — |
| 6 | 64 | 32 | ReLU | — |
| 7 (çıkış) | 32 | 4 | — | — |

**Optimizer:** Adam (lr=0.001, weight_decay=1e-5)  
**Kayıp:** L1Loss (MAE)  
**Scheduler:** ReduceLROnPlateau (factor=0.5, patience=5)  
**Erken Durdurma:** patience=15 epoch

### Neden L1Loss (MAE) ?
Formasyon enerjisi dağılımı aykırı değerler içerir.  
L1Loss, MSE'ye kıyasla aykırı değerlere karşı **daha dayanıklıdır** (robust).  
Bu, modelin ortalama tahmin kalitesini aykırı değerler pahasına feda etmemesini sağlar.